In [1]:
import warnings

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
)

## Enable if upsampling is used 
# from sklearn.utils import resample 
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
import pandas as pd
import numpy as np
import os
import torch
import json


## GPU/RAM Check

In [2]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Thu Mar 19 10:42:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   36C    P0             49W /  600W |       3MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 189.9 gigabytes of available RAM

You are using a high-RAM runtime!


## Preparing Training Data

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
human_labeled_sentiment = pd.read_parquet('/content/drive/MyDrive/ColabThesisData/labeled.parquet')

# Replace with your dataset loading
ds = human_labeled_sentiment

In [6]:
ds.head(10)

,id,author_id,message,date_time,engagement,forum,url,company_name,ticker,message_length,year_month,year,sentiment,labeled_at
0,Sijoitustieto.comment-117218,Sijoitustieto.Unknown,"Juuri näin. Yllättävää,, että tässä tulee jatk...",2025-03-01 03:19:00,N/A,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,Citycon,CTY1S,109,2025-03,2025,1,2026-03-16T00:36:15.908780
1,Sijoitustieto.comment-26243,Sijoitustieto.Unknown,Tervehdys! \n\nMukava kuulla. Tässä tullut lis...,2019-06-08 19:37:00,N/A,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,Saga Furs,SAGCV,136,2019-06,2019,2,2026-03-16T00:36:23.403400
2,Kauppalehti.post-6085573,Kauppalehti.55430,Siis Hannu Lehto antoi ymmärtää. Vaikeudet on ...,2019-12-21 19:49:03,N/A,Kauppalehti,https://keskustelu.kauppalehti.fi/threads/leht...,Lehto Group,LEHTO,197,2019-12,2019,2,2026-03-16T00:36:33.661733
3,Sijoitustieto.comment-24835,Sijoitustieto.Unknown,Ihan kiinnostavalta vaikuttaa. IPOihin en ole ...,2019-03-23 17:10:00,N/A,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,Aallon Group,AALLON,216,2019-03,2019,2,2026-03-16T00:36:42.800170
4,Kauppalehti.post-6189721,Kauppalehti.55017,"Hallitushan allokaatiosta toki päättää, mutta ...",2016-09-28 18:01:56,N/A,Kauppalehti,https://keskustelu.kauppalehti.fi/threads/vinc...,Vincit,VINCIT,265,2016-09,2016,1,2026-03-16T00:36:53.395512
5,Kauppalehti.post-6063559,Kauppalehti.981,YK jäi valitettavasti väliin mutta kiitokset s...,2017-03-27 09:57:47,N/A,Kauppalehti,https://keskustelu.kauppalehti.fi/threads/reve...,Revenio,REG1V,2085,2017-03,2017,2,2026-03-16T09:31:22.161538
6,Sijoitustieto.comment-109908,Sijoitustieto.Unknown,Omat käynnit Tokmannilla liittyvät käytännössä...,2024-07-24 12:58:00,+4,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,Tokmanni,TOKMAN,360,2024-07,2024,1,2026-03-16T09:37:35.563658
7,Kauppalehti.post-7398490,Kauppalehti.51236,Jiiärrä sanoi:\nElä hyvä mies lähde nesteeseen...,2024-05-08 07:27:55,N/A,Kauppalehti,https://keskustelu.kauppalehti.fi/threads/kone...,Konecranes,KCR,195,2024-05,2024,1,2026-03-16T09:38:39.687686
8,Sijoitustieto.comment-26007,Sijoitustieto.Unknown,Harvia tykitti muuten aika timmin tuloksen. Tä...,2019-05-22 19:01:00,N/A,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,Harvia,HARVIA,514,2019-05,2019,2,2026-03-16T09:39:00.230697
9,Kauppalehti.post-5666319,Kauppalehti.45987,> Lehdistötiedote:\n>\n> Biohit Oyj tuo markki...,2016-11-16 07:08:15,N/A,Kauppalehti,https://keskustelu.kauppalehti.fi/threads/mita...,Biohit,BIOBV,628,2016-11,2016,1,2026-03-16T09:39:19.286718


In [7]:
ds["sentiment"].value_counts()

,count
sentiment,
1,253
2,205
0,150


## Load Pretrained Model

In [8]:
MODEL_PATH = 'FacebookAI/xlm-roberta-large'
NUM_LABELS = 3
LABEL_NAMES = ["negative", "neutral", "positive"]
MAX_SEQ_LENGTH = 512  # XLM-RoBERTa max is 512 tokens

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=NUM_LABELS,
    device_map="auto",
    dtype=torch.bfloat16,
)

model.eval()
print(f"Model loaded from: {MODEL_PATH}")
print(f"Model device: {next(model.parameters()).device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded from: FacebookAI/xlm-roberta-large
Model device: cuda:0


## Fine-tune on Financial Domain Data

In [9]:
# 80/20 stratified split — holdout never seen during fine-tuning
ft_train_df, ft_holdout_df = train_test_split(
    ds[["message", "sentiment", "company_name"]],
    test_size=0.2,
    random_state=42,
    stratify=ds["sentiment"],
)

print(f"Fine-tune train: {len(ft_train_df)}  |  Holdout: {len(ft_holdout_df)}")
print("Train distribution:\n", ft_train_df["sentiment"].value_counts().sort_index())
print("Holdout distribution:\n", ft_holdout_df["sentiment"].value_counts().sort_index())

Fine-tune train: 486  |  Holdout: 122
Train distribution:
 sentiment
0    120
1    202
2    164
Name: count, dtype: int64
Holdout distribution:
 sentiment
0    30
1    51
2    41
Name: count, dtype: int64


In [10]:
def make_hf_dataset(df):
    df = df.copy()
    df["text"] = "yritys: " + df["company_name"] + " viesti: " + df["message"]
    hf = Dataset.from_pandas(df[["text", "sentiment"]].rename(columns={"sentiment": "label"}).reset_index(drop=True))
    return hf.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_SEQ_LENGTH),
        batched=True,
    )

ft_train_ds   = make_hf_dataset(ft_train_df)
ft_holdout_ds = make_hf_dataset(ft_holdout_df)

Map:   0%|          | 0/486 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

In [11]:
from torch import nn
from sklearn.utils.class_weight import compute_class_weight

FINETUNED_MODEL_PATH = '/content/drive/MyDrive/ColabThesisData/final_model_finetuned/'

# Compute class weights from the training split to counter neutral-class dominance
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=ft_train_df['sentiment'].values,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(model.device)
print(f"Class weights: neg={class_weights[0]:.3f}  neu={class_weights[1]:.3f}  pos={class_weights[2]:.3f}")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall": recall_score(labels, preds, average="weighted", zero_division=0),
    }

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=class_weights_tensor)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

ft_training_args = TrainingArguments(
    output_dir='/tmp/ft_checkpoints/',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,           # 10% of total steps — more robust than a fixed count
    logging_steps=5,            # small dataset: log every 5 steps to see training loss
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    bf16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=True,
)

ft_trainer = WeightedTrainer(
    model=model,
    args=ft_training_args,
    train_dataset=ft_train_ds,
    eval_dataset=ft_holdout_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

ft_trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Class weights: neg=1.350  neu=0.802  pos=0.988


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.116661,1.100958,0.262295,0.129623,0.397541,0.262295
2,1.117015,1.100758,0.213115,0.137211,0.117657,0.213115
3,1.117964,1.101031,0.336066,0.226629,0.271776,0.336066
4,1.130422,1.104574,0.327869,0.205810,0.338728,0.327869
5,1.109138,1.106112,0.303279,0.198395,0.201600,0.303279
6,1.090921,1.109364,0.311475,0.220219,0.269149,0.311475
7,1.115098,1.101521,0.311475,0.211021,0.383516,0.311475
8,1.119374,1.102875,0.360656,0.307224,0.459330,0.360656
9,1.105222,1.104116,0.319672,0.251989,0.362377,0.319672
10,1.090474,1.104981,0.327869,0.255610,0.359471,0.327869


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=310, training_loss=1.1041658832180885, metrics={'train_runtime': 117.5741, 'train_samples_per_second': 41.336, 'train_steps_per_second': 2.637, 'total_flos': 3495289554750636.0, 'train_loss': 1.1041658832180885, 'epoch': 10.0})

In [12]:
# Save fine-tuned model
ft_trainer.save_model(FINETUNED_MODEL_PATH)
tokenizer.save_pretrained(FINETUNED_MODEL_PATH)
print(f"Fine-tuned model saved to: {FINETUNED_MODEL_PATH}")

# Evaluate on holdout
holdout_results = ft_trainer.predict(ft_holdout_ds)
ft_preds = np.argmax(holdout_results.predictions, axis=1)
ft_true = ft_holdout_df["sentiment"].tolist()

print("\n" + "=" * 50)
print("HOLDOUT RESULTS AFTER FINE-TUNING")
print("=" * 50)
print(classification_report(ft_true, ft_preds, target_names=LABEL_NAMES))

print("Confusion matrix (rows=true, cols=predicted):")
print(pd.DataFrame(
    confusion_matrix(ft_true, ft_preds),
    index=[f"true_{l}" for l in LABEL_NAMES],
    columns=[f"pred_{l}" for l in LABEL_NAMES],
))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved to: /content/drive/MyDrive/ColabThesisData/final_model_finetuned/



HOLDOUT RESULTS AFTER FINE-TUNING
              precision    recall  f1-score   support

    negative       0.25      0.07      0.11        30
     neutral       0.60      0.18      0.27        51
    positive       0.32      0.78      0.46        41

    accuracy                           0.35       122
   macro avg       0.39      0.34      0.28       122
weighted avg       0.42      0.35      0.29       122

Confusion matrix (rows=true, cols=predicted):
               pred_negative  pred_neutral  pred_positive
true_negative              2             1             27
true_neutral               2             9             40
true_positive              4             5             32


## Augmented Training: Human + LLM Labels

In [13]:
LLM_LABELS_PATH = '/content/drive/MyDrive/ColabThesisData/llm_labeled.parquet'

llm_labels = pd.read_parquet(LLM_LABELS_PATH)

# Safety: recover holdout IDs from the original ds index and exclude them
holdout_ids = set(ds.loc[ft_holdout_df.index, 'id'])
leaked = llm_labels[llm_labels['id'].isin(holdout_ids)]
if len(leaked):
    print(f"WARNING: {len(leaked)} LLM labels overlap with holdout — dropping them.")
    llm_labels = llm_labels[~llm_labels['id'].isin(holdout_ids)]

# Combine: human train split + all LLM labels (keep company_name for text formatting)
ft_train_augmented_df = pd.concat(
    [ft_train_df, llm_labels[["message", "sentiment", "company_name"]]],
    ignore_index=True,
)

print(f"Train  : {len(ft_train_df)} human  →  {len(ft_train_augmented_df)} total (+{len(llm_labels)} LLM)")
print(f"Holdout: {len(ft_holdout_df)} (unchanged)")
print("\nAugmented train sentiment distribution:")
print(ft_train_augmented_df["sentiment"].value_counts().sort_index().rename({0: "negative", 1: "neutral", 2: "positive"}))

Train  : 486 human  →  10486 total (+10000 LLM)
Holdout: 122 (unchanged)

Augmented train sentiment distribution:
sentiment
negative    3711
neutral     2984
positive    3791
Name: count, dtype: int64


In [14]:
# Reload base model fresh — same starting point as before, not the already fine-tuned weights
model_aug = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=NUM_LABELS,
    device_map="auto",
    dtype=torch.bfloat16,
)

ft_train_augmented_ds = make_hf_dataset(ft_train_augmented_df)
# ft_holdout_ds is unchanged

AUGMENTED_MODEL_PATH = '/content/drive/MyDrive/ColabThesisData/final_model_finetuned_augmented/'

aug_training_args = TrainingArguments(
    output_dir='/tmp/aug_checkpoints/',
    num_train_epochs=15,         # more data needs more epochs to converge
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,          # slightly higher — larger dataset can support it
    weight_decay=0.01,
    warmup_steps=250,
    logging_steps=50,            # log training loss every 50 steps so we can see it move
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    bf16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=True,
)

aug_trainer = Trainer(
    model=model_aug,
    args=aug_training_args,
    train_dataset=ft_train_augmented_ds,
    eval_dataset=ft_holdout_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

aug_trainer.train()

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/10486 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.874396,0.977407,0.581967,0.565860,0.646535,0.581967
2,0.767206,0.952877,0.573770,0.565926,0.601670,0.573770
3,0.701665,0.979276,0.590164,0.575312,0.633449,0.590164
4,0.650724,1.006871,0.598361,0.582887,0.641368,0.598361
5,0.563203,1.025401,0.631148,0.627881,0.678571,0.631148
6,0.487365,1.102404,0.622951,0.619904,0.657973,0.622951
7,0.445615,1.104433,0.614754,0.608603,0.661286,0.614754
8,0.464528,1.145824,0.639344,0.645561,0.678503,0.639344
9,0.423121,1.182652,0.622951,0.615394,0.666487,0.622951
10,0.423464,1.223318,0.614754,0.610538,0.649545,0.614754


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=9840, training_loss=0.5171872562509242, metrics={'train_runtime': 988.9462, 'train_samples_per_second': 159.048, 'train_steps_per_second': 9.95, 'total_flos': 1.0814022828002709e+17, 'train_loss': 0.5171872562509242, 'epoch': 15.0})

In [15]:
aug_trainer.save_model(AUGMENTED_MODEL_PATH)
tokenizer.save_pretrained(AUGMENTED_MODEL_PATH)
print(f"Augmented model saved to: {AUGMENTED_MODEL_PATH}")

aug_results = aug_trainer.predict(ft_holdout_ds)
aug_preds = np.argmax(aug_results.predictions, axis=1)
ft_true   = ft_holdout_df["sentiment"].tolist()

print("\n" + "=" * 50)
print("HOLDOUT RESULTS — AUGMENTED (human + LLM)")
print("=" * 50)
print(classification_report(ft_true, aug_preds, target_names=LABEL_NAMES))

print("Confusion matrix (rows=true, cols=predicted):")
print(pd.DataFrame(
    confusion_matrix(ft_true, aug_preds),
    index=[f"true_{l}" for l in LABEL_NAMES],
    columns=[f"pred_{l}" for l in LABEL_NAMES],
))

# Side-by-side summary vs the human-only run
print("\n── Accuracy comparison ──")
print(f"  Human only  (~486 train) : {accuracy_score(ft_true, ft_preds):.3f}")
print(f"  Human + LLM (~{len(ft_train_augmented_df)} train) : {accuracy_score(ft_true, aug_preds):.3f}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Augmented model saved to: /content/drive/MyDrive/ColabThesisData/final_model_finetuned_augmented/



HOLDOUT RESULTS — AUGMENTED (human + LLM)
              precision    recall  f1-score   support

    negative       0.47      0.73      0.57        30
     neutral       0.78      0.55      0.64        51
    positive       0.74      0.71      0.72        41

    accuracy                           0.65       122
   macro avg       0.66      0.66      0.65       122
weighted avg       0.69      0.65      0.65       122

Confusion matrix (rows=true, cols=predicted):
               pred_negative  pred_neutral  pred_positive
true_negative             22             5              3
true_neutral              16            28              7
true_positive              9             3             29

── Accuracy comparison ──
  Human only  (~486 train) : 0.352
  Human + LLM (~10486 train) : 0.648
